# Replacement Trap Analysis - Core Calculations

This notebook builds the core analysis DataFrame via `lib/replacement_trap_pipeline.py`
(shared with `scripts/regenerate_core_outputs.py`).

**Dependencies:**
- `lib/replacement_trap_pipeline.py` - Shared Scenario A/B DataFrame builder
- `lib/replacement_trap_config.py` - Configuration and constants
- `lib/replacement_trap_utils.py` - Calculation helpers
- `data/systems-data.json` - System data

**Output:** DataFrame `df` with Scenario A (replace at expected lifespan) and
Scenario B (replace at warranty end) metrics, ready for visualization and Monte Carlo.

**Methodology Note:** Analysis uses undiscounted cash flows for accessibility and worst-case conservatism.



In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import importlib

# Add lib directory to path for imports
cwd = Path.cwd()
lib_dir = None

if (cwd.parent / 'lib' / 'replacement_trap_utils.py').exists() and cwd.name == 'analysis':
    lib_dir = cwd.parent / 'lib'
elif (cwd / 'lib' / 'replacement_trap_utils.py').exists():
    lib_dir = cwd / 'lib'
elif (cwd / 'notebooks' / 'lib' / 'replacement_trap_utils.py').exists():
    lib_dir = cwd / 'notebooks' / 'lib'
else:
    for parent in [cwd] + list(cwd.parents):
        if (parent / 'lib' / 'replacement_trap_utils.py').exists():
            lib_dir = parent / 'lib'
            break

if lib_dir is None and cwd.name == 'analysis':
    potential_lib = cwd.parent / 'lib'
    if (potential_lib / 'replacement_trap_utils.py').exists():
        lib_dir = potential_lib

if lib_dir is None:
    raise ImportError(
        f"Could not locate lib directory. Current working directory: {cwd}\n"
        f"Expected to find lib/replacement_trap_utils.py relative to project root."
    )

if str(lib_dir) not in sys.path:
    sys.path.insert(0, str(lib_dir))

import replacement_trap_utils as rtu
rtu = importlib.reload(rtu)
get_notebooks_root = rtu.get_notebooks_root

notebooks_root = get_notebooks_root()
lib_dir = notebooks_root / 'lib'
if str(lib_dir) not in sys.path:
    sys.path.insert(0, str(lib_dir))

from replacement_trap_config import (
    ELECTRICITY_RATE,
    WATER_RATE,
    HELOC_RATE,
    HELOC_LOAN_TERM,
    TIME_HORIZON,
    EFFICIENCY_DEGRADATION_RATE,
)
from replacement_trap_utils import (
    calculate_lifetime_value_cash,
    calculate_lifetime_value_heloc,
    validate_scenario,
    load_reference_and_compare,
)
from replacement_trap_pipeline import build_core_dataframe, build_reference_outputs

try:
    from replacement_trap_validation import validate_unit_consistency
    from validate_reference_data import validate_systems_data
    VALIDATION_AVAILABLE = True
except ImportError:
    VALIDATION_AVAILABLE = False
    print("Validation modules not available - skipping validation checks")

print("✓ Imports loaded")
print(f"  Electricity rate: ${ELECTRICITY_RATE:.4f}/kWh")
print(f"  Water rate: ${WATER_RATE:.5f}/gal")
print(f"  HELOC rate: {HELOC_RATE*100:.1f}%, term: {HELOC_LOAN_TERM} years")
print(f"  Time horizon: {TIME_HORIZON} years")

if VALIDATION_AVAILABLE:
    print("\nRunning unit consistency validation...")
    unit_result = validate_unit_consistency()
    if unit_result.passed:
        print("  ✓ Unit consistency checks passed")
    else:
        print(f"  Unit consistency checks failed: {len(unit_result.errors)} error(s)")
        for err in unit_result.errors[:3]:
            print(f"    - {err}")
    if unit_result.warnings:
        print(f"  {len(unit_result.warnings)} warning(s) - review recommended")


## 1. Build Core Analysis DataFrame

Load `data/systems-data.json` and compute metrics via `build_core_dataframe()`:

- **Scenario A:** emergency replacement at expected lifespan
- **Scenario B:** proactive replacement at warranty expiration (shorter effective life → worse R/P)

Also computes payback, comfort gap, cash vs HELOC lifetime values, and financing amplification.


In [ ]:
df = build_core_dataframe()

print(f"Loaded {len(df)} systems")
print(f"  Categories: {df['category'].unique().tolist()}")
print(f"  DataFrame shape: {df.shape}")

assert len(df) == 11, f"Expected 11 systems, got {len(df)}"
assert df['category'].nunique() == 5, f"Expected 5 categories, got {df['category'].nunique()}"
assert (df['annual_savings'] > 0).all(), "All annual savings should be positive"
assert (df['warranty_years'] <= df['lifespan_years']).all(), "Warranty should not exceed lifespan"
assert (df['rp_ratio_scenario_b'] <= df['rp_ratio_scenario_a'] + 1e-12).all(), \
    "Scenario B R/P should not exceed Scenario A when warranty <= lifespan"

rp_stats = df['rp_ratio_scenario_a'].agg(['min', 'max', 'mean'])
structural_losses = (df['rp_ratio_scenario_a'] < 0.8).sum()
surplus_generators = (df['rp_ratio_scenario_a'] > 1.0).sum()
print(f"✓ Scenario A R/P range: {rp_stats['min']:.3f} to {rp_stats['max']:.3f} (mean {rp_stats['mean']:.3f})")
print(f"  Structural loss (R/P < 0.8): {structural_losses}; surplus (R/P > 1.0): {surplus_generators}")
print(f"✓ Scenario B R/P mean: {df['rp_ratio_scenario_b'].mean():.3f}")
print(f"  Cash mean (A): ${df['lifetime_value_cash_scenario_a'].mean():,.0f}")
print(f"  Cash mean (B): ${df['lifetime_value_cash_scenario_b'].mean():,.0f}")

if VALIDATION_AVAILABLE:
    print("\nRunning reference data validation...")
    data_result = validate_systems_data()
    if data_result.passed:
        print("  ✓ Reference data validation passed")
    else:
        print(f"  Reference data validation failed: {len(data_result.errors)} error(s)")
        for err in data_result.errors[:3]:
            print(f"    - {err}")

df.head()


## 2. Comfort Gap Analysis

Comfort gap is computed in the shared pipeline. Display environmental utilities only
(water heaters, AC, insulation).


In [ ]:
comfort_categories = ['Water Heater', 'Air Conditioner', 'Attic Insulation']
environmental_utilities = df[df['category'].isin(comfort_categories)].copy()

if len(environmental_utilities) > 0:
    print("COMFORT GAP ANALYSIS")
    print("=" * 80)
    print(f"Environmental utilities analyzed: {len(environmental_utilities)}")
    print(
        f"Comfort gap range: "
        f"${environmental_utilities['comfort_gap'].min():,.2f} to "
        f"${environmental_utilities['comfort_gap'].max():,.2f}/year"
    )
    print(f"Average comfort gap: ${environmental_utilities['comfort_gap'].mean():,.2f}/year")
    print(f"Average R/P ratio (A): {environmental_utilities['rp_ratio_scenario_a'].mean():.3f}")


## 3. Validation

Run validation checks on calculated metrics.



In [ ]:
# Validate scenarios for each system
print("=" * 80)
print("VALIDATION: Scenario Results")
print("=" * 80)

for idx, row in df.iterrows():
    scenario_result = {
        'payback_years': row['payback_period'],
        'lifespan': row['lifespan_years'],
        'annual_savings': row['annual_savings'],
        'upfront_cost': row['installed_cost'],
        'net_value': row['lifetime_value_cash_scenario_a'],
        'total_savings': row['lifetime_value_cash_scenario_a'] + row['installed_cost']  # Approximate
    }
    try:
        validate_scenario(scenario_result, f"{row['category']} - {row['model']}")
    except AssertionError as e:
        print(f"⚠ Validation warning: {e}")

print(f"\n✓ Validated {len(df)} systems")
print(f"  All payback periods valid")
print(f"  All R/P ratios non-negative")
print(f"  All annual savings positive")

# Compare against full golden reference outputs
try:
    current_outputs = build_reference_outputs(df)
    load_reference_and_compare(current_outputs)
    print("✓ Outputs match reference (full golden payload)")
except FileNotFoundError:
    print("Reference outputs file not found - skipping regression test")
except AssertionError as e:
    print(f"Regression test failed: {e}")

print("\n" + "=" * 80)
print("✓ Core analysis complete")
print("=" * 80)
print(f"\nDataFrame ready with {len(df)} systems and all calculated metrics.")
print("Use this DataFrame for visualization and Monte Carlo analysis.")



In [ ]:
# Validation: Trane XV18 cash vs HELOC (undiscounted vs discounted)
def _format_currency(value):
    sign = "-" if value < 0 else ""
    return f"{sign}${abs(value):,.2f}"


def _npv_penalty(npv_value, undiscounted_value):
    if np.isclose(undiscounted_value, 0, atol=1e-9):
        return np.nan
    return (abs(npv_value) - abs(undiscounted_value)) / abs(undiscounted_value) * 100


def _print_validation_block(name, cumulative_value, npv_value, penalty_pct):
    print(f"{name}:")
    print(f"  - 30-year cumulative (undiscounted): {_format_currency(cumulative_value)}")
    print(f"  - 30-year NPV (4% discount): {_format_currency(npv_value)}")
    if np.isnan(penalty_pct):
        print("  - NPV impact: --")
    elif penalty_pct >= 0:
        print(f"  - NPV makes it {penalty_pct:.1f}% worse (larger magnitude loss)")
    else:
        print(f"  - NPV makes it {abs(penalty_pct):.1f}% less severe (discounting shrinks losses)")
    print()


target_model = "Trane XV18 Variable-Speed Air Conditioner"
trane_row = df[df['model'] == target_model].iloc[0]

cash_flow_trane, cash_npv_trane = calculate_lifetime_value_cash(
    trane_row['installed_cost'],
    trane_row['annual_savings'],
    trane_row['lifespan_years'],
    horizon=TIME_HORIZON,
    degradation_rate=EFFICIENCY_DEGRADATION_RATE,
)
heloc_flow_trane, heloc_npv_trane = calculate_lifetime_value_heloc(
    trane_row['installed_cost'],
    trane_row['annual_savings'],
    trane_row['lifespan_years'],
    rate=HELOC_RATE,
    loan_term=HELOC_LOAN_TERM,
    horizon=TIME_HORIZON,
    degradation_rate=EFFICIENCY_DEGRADATION_RATE,
)

cash_cumulative_trane = cash_flow_trane[-1]
heloc_cumulative_trane = heloc_flow_trane[-1]

cash_penalty_pct_trane = _npv_penalty(cash_npv_trane, cash_cumulative_trane)
heloc_penalty_pct_trane = _npv_penalty(heloc_npv_trane, heloc_cumulative_trane)

print("Trane XV18 Variable-Speed Air Conditioner\n")
_print_validation_block("Cash Purchase", cash_cumulative_trane, cash_npv_trane, cash_penalty_pct_trane)
_print_validation_block("HELOC Financing", heloc_cumulative_trane, heloc_npv_trane, heloc_penalty_pct_trane)

if cash_penalty_pct_trane < 0:
    print("Note: At 4% discounting, the far-future replacement hits (years 15 and 30) shrink faster than the small annual savings,"
          " so the discounted view looks less severe than the raw cumulative loss.\n")
elif cash_penalty_pct_trane > 0:
    print("Note: Discounting magnifies losses because the near-term cash outlay dominates the small, discounted savings stream.\n")



In [ ]:
from pathlib import Path

if 'get_notebooks_root' not in globals():
    def get_notebooks_root():
        """Fallback path resolver so this cell can run without the setup cell."""
        cwd = Path.cwd()

        if (cwd.parent / 'lib' / 'replacement_trap_utils.py').exists() and cwd.name == 'analysis':
            return cwd.parent
        if (cwd / 'lib' / 'replacement_trap_utils.py').exists():
            return cwd
        if (cwd / 'notebooks' / 'lib' / 'replacement_trap_utils.py').exists():
            return cwd / 'notebooks'
        for parent in [cwd] + list(cwd.parents):
            if (parent / 'lib' / 'replacement_trap_utils.py').exists():
                return parent

        return cwd.parent if cwd.name == 'analysis' else cwd

# Save DataFrame for use in other notebooks
notebooks_root = get_notebooks_root()
df_save_path = notebooks_root / 'data' / 'replacement_trap_df.pkl'
df.to_pickle(df_save_path)
print(f"DataFrame saved to {df_save_path}")
print(f"  Shape: {df.shape}")
print(f"  Columns: {len(df.columns)}")
